# 01 — Time Series Decomposition (STL)

**Objective:** Analyse electricity price structure BEFORE any external features.
Demonstrates the 4 structural components and motivates the need for exogenous drivers.

Components: **Observed | Trend | Seasonal | Residual**

In [1]:
import os, sys, warnings
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['font.size'] = 10
plt.rcParams['axes.unicode_minus'] = False
import matplotlib.gridspec as gridspec
from statsmodels.tsa.seasonal import STL
warnings.filterwarnings('ignore')
# Run from notebooks/ directory
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working dir:', os.getcwd())

Working dir: D:\Projects\data\ember


In [2]:
COUNTRIES = ['DE', 'DK', 'ES', 'FR', 'IT', 'PL']
TARGET    = 'Real_Wholesale_Price_EUR'
COLORS    = {'DE':'#2c3e50','DK':'#e74c3c','ES':'#f39c12',
             'FR':'#3498db','IT':'#27ae60','PL':'#8e44ad'}
OUT_DIR   = 'reports/figures/00_ts_decomposition'
os.makedirs(OUT_DIR, exist_ok=True)
plt.rcParams.update({
    'font.size':10, 'axes.labelsize':11, 'axes.titlesize':12,
    'figure.titlesize':14, 'axes.spines.top':False, 'axes.spines.right':False
})

In [3]:
df = pd.read_csv('data/processed/electricity_dataset.csv')
df['Datetime'] = pd.to_datetime(df['Datetime'], utc=True)
df = df.sort_values(['Country','Datetime']).reset_index(drop=True)
print(f'{len(df):,} rows loaded')

420,768 rows loaded


## 1. STL Decomposition per Country (4-panel plot)

In [4]:
def plot_stl_country(result, country, series, out_dir, save=True):
    fig = plt.figure(figsize=(16, 12))
    gs  = gridspec.GridSpec(4, 1, hspace=0.5)
    dates = series.index
    crisis_start = pd.Timestamp('2021-09-01', tz='UTC')
    crisis_end   = pd.Timestamp('2023-01-01', tz='UTC')
    panels = [
        ('Observed (Actual Price)',                  series.values,   '#2c3e50'),
        ('Trend (Long-term Direction)',              result.trend,    '#e74c3c'),
        ('Seasonal (Annual Pattern)',                result.seasonal, '#3498db'),
        ('Residual (Unexplained — Crisis Spikes)',   result.resid,    '#e67e22'),
    ]
    for i, (title, data, color) in enumerate(panels):
        ax = fig.add_subplot(gs[i])
        ax.plot(dates, data, color=color, lw=0.8, alpha=0.9)
        ax.axvspan(crisis_start, crisis_end, color='#e74c3c', alpha=0.08,
                   label='Crisis 2021-2022' if i==0 else '')
        ax.axhline(0, color='grey', ls='--', lw=0.5, alpha=0.5)
        ax.set_title(title, fontweight='bold', loc='left')
        ax.set_ylabel('EUR/MWh')
        if i == 3:
            max_idx  = int(np.argmax(np.abs(result.resid)))
            max_val  = float(result.resid.iloc[max_idx])
            max_date = dates[max_idx]
            ax.annotate(
                f'Max residual:\n{max_val:.0f} EUR/MWh\n({max_date.strftime("%Y-%m")})',
                xy=(max_date, max_val), xytext=(20, -30),
                textcoords='offset points',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
                fontsize=9, bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8)
            )
    fig.suptitle(f'{country} — STL Decomposition (2018-2025)', fontsize=14, fontweight='bold')
    if save:
        path = os.path.join(out_dir, f'{country}_stl.png')
        plt.savefig(path, dpi=500, bbox_inches='tight')
        print(f'  Saved: {path}')
    plt.show()

In [5]:
from joblib import Parallel, delayed

def run_stl_for_country(country):
    print(f'[{country}] unning STL...')
    df_c   = df[df['Country'] == country].copy()
    series = df_c.set_index('Datetime')[TARGET].resample('D').mean().interpolate()
    result = STL(series, period=365, seasonal=51, seasonal_deg=0, robust=True).fit()
    plot_stl_country(result, country, series, OUT_DIR)
    return country, result, series

# un in parallel across 6 cores
results = Parallel(n_jobs=1)(delayed(run_stl_for_country)(c) for c in COUNTRIES)
stl_results = {r[0]: (r[1], r[2]) for r in results}


[DE] unning STL...


  Saved: reports/figures/00_ts_decomposition\DE_stl.png
[DK] unning STL...


  Saved: reports/figures/00_ts_decomposition\DK_stl.png
[ES] unning STL...


  Saved: reports/figures/00_ts_decomposition\ES_stl.png
[FR] unning STL...


  Saved: reports/figures/00_ts_decomposition\FR_stl.png
[IT] unning STL...


  Saved: reports/figures/00_ts_decomposition\IT_stl.png
[PL] unning STL...


  Saved: reports/figures/00_ts_decomposition\PL_stl.png


## 2. Trend Comparison — All 6 Countries

In [6]:
fig, ax = plt.subplots(figsize=(16, 6))
for country in COUNTRIES:
    result, series = stl_results[country]
    trend_s = pd.Series(result.trend, index=series.index).resample('7D').mean()
    ax.plot(trend_s.index, trend_s.values, label=country, color=COLORS[country], lw=1.8)
ax.axvspan(pd.Timestamp('2021-09-01',tz='UTC'), pd.Timestamp('2023-01-01',tz='UTC'),
           color='red', alpha=0.07, label='Energy Crisis')
ax.set_title('Price Trend (Weekly Smoothed) — 6 Countries', fontsize=14, fontweight='bold')
ax.set_ylabel('EUR/MWh')
ax.legend(ncol=7, loc='upper left', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, 'all_countries_trend.png'), dpi=500, bbox_inches='tight')
plt.show()

## 3. Paper Figures: Seasonal & Residual Grid (3×2)

Tạo ảnh đa quốc gia ghép 3×2 cho từng thành phần STL, xuất sang `paper/figures/methodology/`.
Phong cách nhất quán: mỗi panel = 1 quốc gia, highlight vùng khủng hoảng 2021–22.

In [7]:
# ─── Seasonal Grid 3×2 ────────────────────────────────────────────────────
import matplotlib.dates as mdates

COLORS_GRID = {
    'DE': '#4d4d4d', 'DK': '#e84545', 'ES': '#f0a500',
    'FR': '#4393c3', 'IT': '#1a9641', 'PL': '#984ea3',
}

fig_s, axes_s = plt.subplots(2, 3, figsize=(18, 8), sharey=False)
fig_s.suptitle(
    'STL Seasonal Component: Annual Price Cycles Across Six Countries\n'
    '(Crisis 2021-2022 highlighted in red)',
    fontsize=13, fontweight='bold'
)

for i, country in enumerate(COUNTRIES):
    ax = axes_s.flatten()[i]
    result, series = stl_results[country]
    seasonal = pd.Series(result.seasonal, index=series.index)
    peak_amp = seasonal.abs().max()

    CRISIS_START = pd.Timestamp('2021-07-01', tz='UTC')
    CRISIS_END   = pd.Timestamp('2022-12-31', tz='UTC')

    ax.fill_between(seasonal.index, seasonal.values, 0,
                    color=COLORS_GRID[country], alpha=0.8)
    ax.plot(seasonal.index, seasonal.values, color=COLORS_GRID[country], lw=0.5, alpha=0.9)
    ax.axvspan(CRISIS_START, CRISIS_END, alpha=0.08, color='#e84545')
    ax.axhline(0, color='black', lw=0.8)
    ax.text(0.02, 0.95, f'Amp: {peak_amp:.0f} EUR/MWh',
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7),
            verticalalignment='top')
    ax.set_title(f'$\\bf{{{country}}}$', fontsize=13)
    ax.set_ylabel('Seasonal (EUR/MWh)', fontsize=8)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
out_seasonal = os.path.join(OUT_DIR, 'all_countries_seasonal.png')
fig_s.savefig(out_seasonal, dpi=500, bbox_inches='tight')

# Copy sang paper figures
import shutil
shutil.copy(out_seasonal, 'paper/figures/methodology/all_countries_seasonal.png')
print(f'Saved: {out_seasonal}')
plt.show()


Saved: reports/figures/00_ts_decomposition\all_countries_seasonal.png


In [8]:
# ─── Residual Crisis Grid 3×2 ─────────────────────────────────────────────
fig_r, axes_r = plt.subplots(2, 3, figsize=(18, 8), sharey=False)
fig_r.suptitle(
    'STL Residual Component: Where Crisis Hides\n(Crisis 2021-2022 highlighted in red)',
    fontsize=13, fontweight='bold'
)

for i, country in enumerate(COUNTRIES):
    ax = axes_r.flatten()[i]
    result, series = stl_results[country]
    residual = pd.Series(result.resid, index=series.index)
    resid_pos = residual.clip(lower=0)
    resid_neg = residual.clip(upper=0)
    peak_max  = residual.max()

    CRISIS_START = pd.Timestamp('2021-07-01', tz='UTC')
    CRISIS_END   = pd.Timestamp('2022-12-31', tz='UTC')

    ax.fill_between(residual.index, residual.values, 0,
                    color=COLORS_GRID[country], alpha=0.8)
    ax.axvspan(CRISIS_START, CRISIS_END, alpha=0.08, color='#e84545')
    ax.axhline(0, color='black', lw=0.8)
    ax.text(0.02, 0.95, f'Max: {peak_max:.0f} EUR/MWh',
            transform=ax.transAxes, fontsize=8,
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7),
            verticalalignment='top')
    ax.set_title(f'$\\bf{{{country}}}$', fontsize=13)
    ax.set_ylabel('Residual (EUR/MWh)', fontsize=8)
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.tick_params(axis='x', labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.96])
out_residual = os.path.join(OUT_DIR, 'all_countries_residual_crisis.png')
fig_r.savefig(out_residual, dpi=500, bbox_inches='tight')
shutil.copy(out_residual, 'paper/figures/methodology/all_countries_residual_crisis.png')
print(f'Saved: {out_residual}')
plt.show()


Saved: reports/figures/00_ts_decomposition\all_countries_residual_crisis.png


## 4. Variance Decomposition Table

In [9]:
rows = []
for country in COUNTRIES:
    result, series = stl_results[country]
    total   = series.var()
    rows.append({
        'Country': country,
        'Trend_%':    round(100 * pd.Series(result.trend).var()    / total, 1),
        'Seasonal_%': round(100 * pd.Series(result.seasonal).var() / total, 1),
        'Residual_%': round(100 * pd.Series(result.resid).var()    / total, 1),
    })
pd.DataFrame(rows).set_index('Country')

,Trend_%,Seasonal_%,Residual_%
Country,,,
DE,14.4,1.4,58.6
DK,13.0,1.8,61.9
ES,32.0,4.2,38.3
FR,24.7,1.9,42.5
IT,22.7,0.7,42.5
PL,27.9,3.1,44.2


## 5. Key Finding

> The **residual component dominates during 2021–2022 crisis** — STL trend and seasonal cannot explain
> the extreme price spikes. This motivates adding exogenous drivers: `TTF_Gas_Price`,
> `EU_Gas_Storage_Anomaly`, and `Residual_Load`.